# 🐕 Dogebone: do DOGE-related posts call the next move?

**Question.** Does the wording of posts from the 10 most influential Dogecoin accounts tell us whether DOGE's
**daily** price goes **up or down** over the next *h* days, and how sure should we be (**conviction**)?

**Method.** An explainable **keyword/lexicon** model (crypto vocabulary, emoji, negation, intensifiers, hedges).
It is **calibrated on 2021** (bubble year) and **tested out-of-sample on 2022** (bear market), and it is scored against no-skill and momentum baselines.

| Section | What it does |
|---|---|
| 0–1 | Setup and configuration knobs |
| 2 | Load and validate prices and posts |
| 3–4 | Text normalization, coin-context tagging, lexicon scoring (with an explainer) |
| 5–6 | Align posts to daily closes without look-ahead → calls: direction + conviction |
| 7 | **Evaluation**: hit rate vs. baselines, by conviction, account, and context |
| 8–11 | Event study, keyword discovery, timeline, illustrative strategy |
| 12–14 | Score your own post, export figures and findings for the presentation, caveats |

> Uses `data/raw/` when real files exist, otherwise `data/sample/` (**synthetic** test data, so don't present those results).
> See `data/README.md` for the accepted file formats.

## 0. Setup

In [ ]:
# Google Colab? Uncomment the next line first:
# !git clone https://github.com/blobberus/dogebone.git && %cd dogebone

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dogebone").exists())
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from dogebone import Config, evaluate, io, lexicon, market, plots, report, signals, text

pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_colwidth", 90)
plots.use_style()
print("Repo root:", ROOT)

## 1. Configuration

Every knob lives in `src/dogebone/config.py`. Override any of them here:

- **`horizons`**: forward windows in days. Crypto trades 7 days a week, so 1 day means the next daily close.
- **`anchor_mode`**: which close a post is measured *from*. `nearest_close` (the default) picks whichever UTC daily close is
  closer in time to the post, so at most 12h of pre-post movement leaks in. `prev_close` captures the full same-day reaction but
  also credits the post for moves that *caused* it. `same_close` is strictly tradeable but misses the immediate reaction.
- **`neutral_threshold`**: posts with |sentiment| below this are *no call*.
- **`train_end`**: everything up to this date is calibration (conviction cutoffs, learned keywords); everything after is the test.

In [ ]:
cfg = Config(
    horizons=(1, 3, 7),
    anchor_mode="nearest_close",
    neutral_threshold=0.05,
    train_end="2021-12-31",
    include_retweets=False,
    price_is_start_of_day_snapshot=False,  # True for CoinGecko 'snapped_at' exports
)
FOCUS_H = 1  # the horizon used for the headline numbers and the presentation
cfg

## 2. Load and validate data

In [ ]:
src = io.resolve_sources(cfg)
DATA_MODE = src["mode"]
if DATA_MODE != "raw":
    display(Markdown("> ## ⚠️ Running on SYNTHETIC sample data\n> Results are for pipeline testing only. "
                     "Put real files in `data/raw/prices/` and `data/raw/tweets/` and re-run."))

prices = io.load_prices(src["doge_price"], cfg)
btc = io.load_prices(src["btc_price"], cfg) if src["btc_price"] else None
posts_all = io.load_tweets(src["tweets"], cfg)
accounts = io.load_accounts(cfg)
events = io.load_events(cfg)

window = (posts_all["created_at"] >= pd.Timestamp(cfg.analysis_start, tz="UTC")) & (
    posts_all["created_at"] < pd.Timestamp(cfg.analysis_end, tz="UTC") + pd.Timedelta(days=1)
)
posts = posts_all[window].reset_index(drop=True)
DATE_ONLY = posts_all.attrs.get("date_only", False)

print(f"Data mode:       {DATA_MODE}")
print(f"DOGE prices:     {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}  ({prices.attrs['n_filled_gaps']} gaps filled)")
print(f"BTC prices:      {'yes (market-adjusted returns enabled)' if btc is not None else 'no'}")
print(f"Posts in window: {len(posts):,} of {len(posts_all):,} loaded  ({posts_all.attrs['n_dropped']} dropped as dupes/retweets)")
print(f"Timestamps:      {'DATE ONLY: intraday anchoring disabled' if DATE_ONLY else 'full time of day'}")
need = pd.Timestamp(cfg.analysis_end) + pd.Timedelta(days=max(cfg.horizons))
if prices.index.max() < need:
    print(f"⚠️  Prices end before {need:%Y-%m-%d}; the last {max(cfg.horizons)}-day outcomes will be missing.")

In [ ]:
# Coverage per account: are all 10 accounts present, and over the full window?
primary = accounts[accounts["tier"] == "primary"]
cov = posts.groupby("username").agg(posts=("text", "size"), first=("created_at", "min"), last=("created_at", "max"))
cov = primary[["rank", "handle", "display_name", "category", "influence_weight"]].merge(
    cov, left_on="handle", right_index=True, how="left"
)
extra = sorted(set(posts["username"]) - set(primary["handle"]))
display(cov.fillna({"posts": 0}))
if (cov["posts"] == 0).any():
    print("⚠️  No posts for:", ", ".join(cov.loc[cov["posts"] == 0, "handle"]))
if extra:
    print("Accounts in data but not in the primary list (default weight applied):", ", ".join(extra))

## 3. Text processing and coin-context tagging

Posts are NFKC-normalized (𝐃𝐎𝐆𝐄 → DOGE), URLs are stripped, elongations are collapsed (*moooon* → *moon*), and emoji are kept as tokens.
Each post is tagged **coin** if it clearly refers to the asset (`$DOGE`, *dogecoin*, *hodl*, exchanges, 🚀, …), otherwise
it gets the account's default context, falling back to **ambiguous** (bare "Doge" memes, dog photos). Ambiguous posts count at
lower conviction (`cfg.context_weights`).

In [ ]:
default_ctx = dict(zip(accounts["handle"], accounts["default_context"]))
tagger = text.ContextTagger(cfg.lexicon_dir / "doge_context.csv", default_ctx)
posts["context"] = [tagger.tag(t, u) for t, u in zip(posts["text"], posts["username"])]

display(posts["context"].value_counts().rename("posts").to_frame())
display(posts.loc[posts["context"] == "ambiguous", ["username", "text"]].head(5))

## 4. Lexicon scoring

The rules follow VADER (Hutto & Gilbert, 2014), with a crypto-specific vocabulary (`lexicons/sentiment_lexicon.csv`, weights from -4 to +4):

- **Greedy longest-phrase match**, so *pump and dump* beats *pump* and *buy the dip* beats *dip*.
- **Negation** within 3 tokens flips the weight and scales it by 0.74 (*not bullish* → bearish).
- **Intensifiers and diminishers** (*very*, *massively*, *slightly*) scale the weight. **ALL-CAPS** terms in a mixed-case post get ×1.25. Each `!` adds emphasis, up to 4.
- **Hedges** (*maybe*, *could*, *nfa*, a trailing `?`) shrink the final score, which lowers conviction.
- The raw sum is normalized to [-1, 1] by x / √(x² + 15).

Edit the CSVs to tune the lexicon; no code changes are needed.

In [ ]:
scorer = lexicon.LexiconScorer(cfg)
scored = scorer.score_frame(posts)

print(f"Lexicon coverage: {(scored['n_hits'] > 0).mean():.0%} of posts contain at least one sentiment term")
top_terms = scored["matched_terms"].str.findall(r"([^,(]+)\(").explode().str.strip().value_counts().head(15)
display(top_terms.rename("posts").to_frame())
cols = ["created_at", "username", "sentiment", "matched_terms", "text"]
display(scored.nlargest(5, "sentiment")[cols])
display(scored.nsmallest(5, "sentiment")[cols])

In [ ]:
# Explainability: why did a post score the way it did?
scorer.explain("Not selling a single $DOGE. Extremely bullish 🚀🚀 but maybe wait for the dip?")

## 5. Align posts to daily prices (no look-ahead)

For each post we find its **anchor close** (see `anchor_mode`) and measure `fwd_ret_h = close[anchor + h] / close[anchor] − 1`.
`mom_h` is the trailing *h*-day return *into* the anchor. It's known at post time and serves as the momentum baseline.
If BTC prices are present we also compute `xs_ret_h`, the DOGE return minus a rolling-β BTC return (the crypto-market move removed).

In [ ]:
scored["anchor_date"] = market.anchor_dates(scored["created_at"], cfg, date_only=DATE_ONLY)
rets = market.forward_returns(prices, cfg.horizons, btc=btc, beta_window=cfg.beta_window)
df = market.attach_returns(scored, rets, cfg.horizons, cfg.flat_band)
df["period"] = signals.split_period(df, "anchor_date", cfg)

base = rets.loc[cfg.analysis_start : cfg.analysis_end].copy()
base["period"] = np.where(base.index <= pd.Timestamp(cfg.train_end), "train", "test")
display(
    base.groupby("period")[[f"fwd_ret_{h}d" for h in cfg.horizons]]
    .agg(lambda s: (s > 0).mean())
    .rename(columns=lambda c: c.replace("fwd_ret_", "share of up-moves, "))
    .style.format("{:.1%}")
)

## 6. Calls: direction and conviction

- **Direction**: UP if sentiment ≥ +threshold, DOWN if ≤ −threshold, otherwise NEUTRAL (no call).
- **Conviction score**: |sentiment| × account influence (`config/accounts.csv`, softened by a square root) × context weight.
- **Conviction bucket**: LOW / MEDIUM / HIGH by terciles **fitted on the training period only**, then applied to the test period.

Because prices are daily, posts sharing an anchor close are also **aggregated into one daily call**: the tanh of the
influence-weighted sentiment sum. Daily calls are the headline unit, since each day has exactly one outcome.

In [ ]:
df = signals.build_post_signals(df, accounts, cfg)
is_call = df["direction"] != "NEUTRAL"
post_cuts = signals.fit_conviction_cutoffs(df.loc[df["period"] == "train", "conviction"], is_call[df["period"] == "train"])
df["conviction_bucket"] = signals.conviction_bucket(df["conviction"], is_call, post_cuts)

daily = signals.build_daily_signals(df, cfg)
daily = market.attach_returns(daily, rets, cfg.horizons, cfg.flat_band)
daily["period"] = signals.split_period(daily, "anchor_date", cfg)
d_call = daily["direction"] != "NEUTRAL"
daily_cuts = signals.fit_conviction_cutoffs(daily.loc[daily["period"] == "train", "conviction"], d_call[daily["period"] == "train"])
daily["conviction_bucket"] = signals.conviction_bucket(daily["conviction"], d_call, daily_cuts)

print("Post-level conviction cutoffs (fit on train):", np.round(post_cuts, 3))
print("Daily conviction cutoffs (fit on train):    ", np.round(daily_cuts, 3))
display(pd.crosstab(daily["period"], daily["direction"], margins=True))

## 7. Evaluation

| Metric | Meaning |
|---|---|
| `hit_rate` | Share of UP/DOWN calls whose direction matched the move (FLAT outcomes excluded) |
| `null_hit_rate` | Expected hit rate if calls were **independent of outcomes**, given how often we called UP and how often price rose. **This is the bar to beat**, not 50%. |
| `edge_pp` / `p_value` | Hit rate minus no-skill, in percentage points; one-sided binomial test |
| `momentum_hit_rate` | Hit rate of "the last *h* days' direction continues". If sentiment doesn't beat this, posts are *echoing* price. |
| `spread_pp` | Mean forward return after UP calls minus after DOWN calls |
| `ic_spearman` | Rank correlation of the continuous score with the forward return |

⚠️ Post-level rows share outcomes (several posts on one day) and multi-day windows overlap, so post-level p-values are optimistic. Prefer the **daily** table.

In [ ]:
fmt = {"coverage": "{:.0%}", "hit_rate": "{:.1%}", "null_hit_rate": "{:.1%}", "edge_pp": "{:+.1f}", "p_value": "{:.3f}",
       "always_up_rate": "{:.1%}", "momentum_hit_rate": "{:.1%}", "mean_ret_when_up": "{:+.2%}",
       "mean_ret_when_down": "{:+.2%}", "spread_pp": "{:+.2f}", "ic_spearman": "{:+.3f}"}

daily_by_period = evaluate.evaluate(daily, cfg.horizons, by="period", score_col="score")
posts_by_period = evaluate.evaluate(df, cfg.horizons, by="period")
display(Markdown("**Daily calls (headline)**"), daily_by_period.style.format(fmt).hide(axis="index"))
display(Markdown("**Post-level calls**"), posts_by_period.style.format(fmt).hide(axis="index"))

if btc is not None:  # same evaluation on BTC-adjusted returns: is the signal DOGE-specific or just crypto beta?
    xs = daily.copy()
    for h in cfg.horizons:
        xs[f"fwd_ret_{h}d"] = xs[f"xs_ret_{h}d"]
        xs[f"actual_{h}d"] = np.where(xs[f"xs_ret_{h}d"] > 0, "UP", np.where(xs[f"xs_ret_{h}d"] < 0, "DOWN", None))
    display(Markdown("**Daily calls vs. BTC-adjusted (excess) returns**"),
            evaluate.evaluate(xs, cfg.horizons, by="period", score_col="score").style.format(fmt).hide(axis="index"))

In [ ]:
# Does conviction mean anything? High-conviction calls should hit more often. Test period only.
test_daily = daily[(daily["period"] == "test") & (daily["conviction_bucket"] != "NONE")]
daily_by_conv_test = evaluate.evaluate(test_daily, cfg.horizons, by="conviction_bucket", score_col="score")
display(daily_by_conv_test.style.format(fmt).hide(axis="index"))
fig_conv = plots.hit_rate_by_conviction(daily_by_conv_test, title="Out-of-sample hit rate by conviction (daily calls)")
plt.show()

In [ ]:
# Which accounts' wording carries signal? (post-level, both periods, focus horizon)
posts_by_account = evaluate.evaluate(df, cfg.horizons, by="username")
acc_view = posts_by_account[posts_by_account["horizon_d"] == FOCUS_H].merge(
    accounts[["handle", "category", "influence_weight"]], left_on="username", right_on="handle", how="left"
).drop(columns="handle").sort_values("edge_pp", ascending=False)
display(acc_view.style.format(fmt).hide(axis="index"))

display(Markdown("**By context**"))
display(evaluate.evaluate(df, [FOCUS_H], by="context").style.format(fmt).hide(axis="index"))

In [ ]:
# Confusion matrix: test period, daily calls, focus horizon
t = daily[(daily["period"] == "test") & daily[f"actual_{FOCUS_H}d"].isin(["UP", "DOWN"])]
pd.crosstab(t["direction"], t[f"actual_{FOCUS_H}d"], rownames=["called"], colnames=[f"actual {FOCUS_H}d"], margins=True)

In [ ]:
# Robustness: is the edge just a few headline event days (SNL, Twitter deal, LUNA, FTX)? Drop ±EXCLUDE_DAYS around each.
EXCLUDE_DAYS = 2
ev_days = set()
for d in events["date"]:
    ev_days.update(pd.date_range(d - pd.Timedelta(days=EXCLUDE_DAYS), d + pd.Timedelta(days=EXCLUDE_DAYS)))
quiet = daily[~daily["anchor_date"].isin(ev_days)]
print(f"Dropped {len(daily) - len(quiet)} of {len(daily)} days within ±{EXCLUDE_DAYS}d of {len(events)} key events")
display(evaluate.evaluate(quiet, cfg.horizons, by="period", score_col="score").style.format(fmt).hide(axis="index"))

## 8. Event study

This is the average cumulative DOGE return from 5 days before to 10 days after each daily call, grouped by direction (0 = anchor close).
Lines that diverge **only after day 0** suggest the posts lead price. Lines that diverge **before day 0** mean posts are *reacting* to moves.

In [ ]:
paths = market.event_paths(rets, daily["anchor_date"], daily["direction"], pre=5, post=10)
fig_event = plots.event_study(paths, title="Average DOGE path around daily calls (all periods)")
plt.show()

## 9. Keyword discovery: let the data pick the words

For every unigram and bigram (and emoji) in **training** posts, we compute the smoothed log-odds that the next move was UP,
relative to the base rate, with a z-score (in the spirit of Monroe et al.'s "Fightin' Words", 2008). Terms with |z| ≥ `learned_z_min`
become a **learned lexicon**, which is then tested on the held-out period against the hand-crafted lexicon.

In [ ]:
train_posts = df[df["period"] == "train"]
term_table = evaluate.term_stats(train_posts, cfg.discovery_horizon, cfg)
display(Markdown(f"**{len(term_table):,} terms with ≥ {cfg.min_term_count} training posts.** Top and bottom by z:"))
display(pd.concat([term_table.head(10), term_table.tail(10)]).style.format(
    {"up_rate": "{:.0%}", "mean_ret": "{:+.2%}", "log_odds": "{:+.2f}", "z": "{:+.2f}"}).hide(axis="index"))
if not term_table.empty:
    fig_kw = plots.keyword_bars(term_table, top=10)
    plt.show()

In [ ]:
learned_w = evaluate.learned_lexicon(term_table, cfg.learned_z_min)
test_posts = df[df["period"] == "test"].copy()
test_posts["learned_score"] = evaluate.score_learned(test_posts["text"], learned_w)
test_posts["learned_direction"] = signals.direction(test_posts["learned_score"], cfg.neutral_threshold)

learned_vs_lexicon = pd.concat([
    evaluate.evaluate(test_posts, cfg.horizons).assign(method="hand-crafted lexicon"),
    evaluate.evaluate(test_posts, cfg.horizons, pred_col="learned_direction", score_col="learned_score").assign(method="learned keywords"),
])
print(f"Learned lexicon size: {len(learned_w)} terms")
display(learned_vs_lexicon.set_index(["method", "horizon_d"]).style.format(fmt))

## 10. Timeline: price, calls, and key events

In [ ]:
fig_timeline = plots.price_with_signals(rets, daily, events, cfg.analysis_start, cfg.analysis_end, max_events=12)
plt.show()

## 11. Illustrative only: following the daily call

Long DOGE for one day after an UP call, short after a DOWN call, and flat otherwise, over the **test period**. There are no costs, slippage,
or funding. This shows the economic size of the signal; it is not a trading recommendation.

In [ ]:
test_start = (pd.Timestamp(cfg.train_end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
eq = evaluate.signal_equity(rets, daily, test_start, cfg.analysis_end)
fig_equity = plots.equity_curves(eq)
plt.show()

## 12. Optional: a generic sentiment baseline (VADER)

Does crypto-specific vocabulary beat an off-the-shelf sentiment model? Run `pip install vaderSentiment` to enable this cell.

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    print("vaderSentiment not installed. Skipping (pip install vaderSentiment).")
else:
    vader = SentimentIntensityAnalyzer()
    tp = test_posts.copy()
    tp["vader"] = tp["text"].map(lambda t: vader.polarity_scores(t)["compound"])
    tp["vader_direction"] = signals.direction(tp["vader"], cfg.neutral_threshold)
    display(pd.concat([
        evaluate.evaluate(tp, cfg.horizons).assign(method="crypto lexicon"),
        evaluate.evaluate(tp, cfg.horizons, pred_col="vader_direction", score_col="vader").assign(method="generic VADER"),
    ]).set_index(["method", "horizon_d"]).style.format(fmt))

## 13. Try it: score any post

In [ ]:
def call_post(post: str, username: str = "elonmusk") -> pd.Series:
    one = pd.DataFrame({"text": [post], "username": [username.lower().lstrip("@")]})
    one["context"] = tagger.tag(post, one["username"][0])
    one = signals.build_post_signals(scorer.score_frame(one), accounts, cfg)
    one["conviction_bucket"] = signals.conviction_bucket(one["conviction"], one["direction"] != "NEUTRAL", post_cuts)
    r = one.iloc[0]
    hist = evaluate.evaluate(df[(df["period"] == "test") & (df["conviction_bucket"] == r["conviction_bucket"])], [FOCUS_H])
    return pd.Series({
        "direction": r["direction"], "conviction": r["conviction_bucket"], "sentiment": round(r["sentiment"], 3),
        "context": r["context"], "matched_terms": r["matched_terms"],
        f"historical {FOCUS_H}d hit rate for this bucket": f"{hist['hit_rate'].iloc[0]:.0%}" if len(hist) and pd.notna(hist['hit_rate'].iloc[0]) else "n/a",
    })

call_post("Dogecoin is the people's crypto 🚀🌕", "elonmusk")

## 14. Export for the presentation

This writes the processed tables to `data/processed/`, the charts to `reports/figures/` and `presentation/figures/`, and an
**auto-drafted `presentation/findings.md`** whose numbers match this run.

In [ ]:
cfg.processed_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(cfg.processed_dir / "posts_scored.csv", index=False)
daily.to_csv(cfg.processed_dir / "daily_signals.csv", index=False)
pd.concat([daily_by_period.assign(level="daily"), posts_by_period.assign(level="post")]).to_csv(
    cfg.processed_dir / "metrics_by_period.csv", index=False)
term_table.to_csv(cfg.processed_dir / "keyword_stats_train.csv", index=False)

figs = {"01_timeline": fig_timeline, "02_hit_rate_by_conviction": fig_conv, "03_event_study": fig_event,
        "05_signal_vs_buy_and_hold": fig_equity}
if not term_table.empty:
    figs["04_keywords"] = fig_kw
fig_dir = cfg.presentation_dir / "figures"
saved = []
mark = None if DATA_MODE == "raw" else "SYNTHETIC DATA"
for name, fig in figs.items():
    plots.save(fig, name, cfg.figures_dir, fig_dir, watermark=mark)
    saved.append(f"presentation/figures/{name}.png")

findings_md = report.write_findings(
    cfg.presentation_dir / "findings.md", cfg, DATA_MODE, FOCUS_H, daily_by_period, daily_by_conv_test,
    posts_by_account, learned_vs_lexicon, term_table, n_posts=len(df), n_accounts=df["username"].nunique(), figures=saved,
)
display(Markdown(findings_md))

## 15. Caveats (read before presenting)

1. **Daily resolution.** A big account's post moves price within minutes; daily closes blur that. `nearest_close` limits the
   overlap with pre-post price action to 12h, but some reverse causality (posting *because* price moved) remains. Compare with the momentum baseline.
2. **Regime shift.** 2021 was a mania (DOGE up >10,000% at the peak) and 2022 a bear market. The base rate of up-days differs a lot, which is
   why we benchmark against `null_hit_rate`, not 50%.
3. **Dependence.** Same-day posts share an outcome and multi-day windows overlap, so post-level p-values are optimistic. Use daily calls.
4. **Lexicon limits.** Sarcasm, memes, and images (a large share of DOGE posts) are invisible to keywords. Explainability is the trade-off we chose.
5. **Selection.** Ten hand-picked accounts, with weights that are judgment calls (`config/accounts.csv`). Learned keywords are fit on 2021 only.
6. **Engagement counts** (likes, reposts) are collected after the fact, so they're excluded from the signal to avoid look-ahead.